# 01 — The Reuter fixed point

Find the non-Gaussian UV fixed point of the Einstein–Hilbert truncation,
compute its critical exponents, and compare to the published Reuter (1998)
values.

**You will see:**
- How to build a `BetaFunctionSystem`
- How `FixedPointFinder` locates the NGFP
- The eigenvalues of the stability matrix → critical exponents `θᵢ`


In [ ]:
import numpy as np
from asymsafety.beta.einstein_hilbert import build_eh_beta_system
from asymsafety.analysis.fixed_points import FixedPointFinder
from asymsafety.analysis.stability import analyze_stability
from asymsafety.validation.reuter_1998 import REUTER_FP


## 1. Build the truncation


In [ ]:
system = build_eh_beta_system(d=4)
system


The system holds the symbolic β-functions for `g` and `lambda`. We can
look at any of them:


In [ ]:
system.beta('g').expression


## 2. Find the NGFP


In [ ]:
finder = FixedPointFinder(system)
fp = finder.find_fixed_point({'g': 0.7, 'lambda': 0.14})
print('NGFP location:', fp.location)


## 3. Analyze stability


In [ ]:
sa = analyze_stability(system, fp)
print(sa.summary())


## 4. Compare to the published Reuter value

The toolkit uses the simplified Litim scheme; values differ from the
published numbers by O(20%) due to gauge / regulator choice. The
qualitative structure (NGFP exists, two relevant directions, λ⋆ < 1/2)
matches Reuter (1998).


In [ ]:
print(f"Toolkit g* = {fp.location['g']:.4f}, λ* = {fp.location['lambda']:.4f}")
print(f"Reuter g* = {REUTER_FP['g_star']:.4f}, λ* = {REUTER_FP['lambda_star']:.4f}")
print(f"Product g*·λ* (toolkit)  = {fp.location['g']*fp.location['lambda']:.4f}")
print(f"Product g*·λ* (Reuter)   = {REUTER_FP['g_star']*REUTER_FP['lambda_star']:.4f}")


## 5. Try a different starting point

All non-Gaussian basins of attraction will converge here:


In [ ]:
for guess in [{'g': 1.5, 'lambda': 0.3}, {'g': 0.3, 'lambda': 0.05}]:
    fp_alt = finder.find_fixed_point(guess)
    print(f'guess={guess} -> {fp_alt.location}')


## 6. Visualisation — canonical phase-portrait + 3D world-line

Two one-liners using the canonical viz API.


In [ ]:
from asymsafety.visualization.phase_portrait import annotated_eh_phase_portrait
from asymsafety.analysis.flow import FlowIntegrator
from asymsafety.gui.visualization_3d import flow_trajectories_3d

fig_2d = annotated_eh_phase_portrait()

trajs = [
    FlowIntegrator(system).integrate(ic, t_span=(-5, 5))
    for ic in [
        {'g': 0.4, 'lambda': 0.05},
        {'g': 0.8, 'lambda': 0.18},
        {'g': 0.3, 'lambda': 0.25},
    ]
]
fig_3d = flow_trajectories_3d(
    trajs, 'g', 'lambda', z_coupling=None,
    fixed_points=[fp], show_eigenvectors=False,
)
